### Run the following code to install packages

In [ ]:
%pip install bitsandbytes==0.47.0
%pip install triton==3.4.0
%pip install torch==2.6.0
%pip install vllm==0.8.1

##### Install Cloud Logging and downgrade the Protocol Buffers packages

In [ ]:
%pip install --upgrade google-cloud-logging
%pip install --force-reinstall protobuf==3.20.0 > /dev/null 2>&1

### Initialize variables and import packages

In [ ]:
from vllm import LLM, SamplingParams
import google.cloud.logging
import logging, time

###Log in HF
import os

#TODO Replace "[TODO - add Hugging Face API token]" with your copied token
os.environ["HUGGING_FACE_HUB_TOKEN"] = "<HF_TOKEN>"

# Define the model and prompts
model_name = "google/gemma-3-4b-it"

system_prompt = "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: "

prompts = [
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: Long waits, waves of calls, web crashes: Social Security is breaking down Lisa Rein and Hannah Natanson, (c) 2025 , The Washington Post Tue, March 25, 2025 Long waits, waves of calls, web crashes: Social Security is breaking down The Social Security Administration website crashed four times in 10 days this month because the servers were overloaded, blocking millions of retirees and disabled Americans from logging in to their online accounts. In the field, office managers have resorted to answering phones in place of receptionists because so many employees have been pushed out. Amid all this, the agency no longer has a system to monitor customer experience because that office was eliminated as part of the cost-cutting efforts led by Elon Musk. And the phones keep ringing. And ringing. Subscribe to The Post Most newsletter for the most important and interesting stories from The Washington Post. The federal agency that delivers $1.5 trillion a year in earned benefits to 73 million retired workers, their survivors, and poor and disabled Americans is engulfed in crisis - further undermining the already struggling organization’s ability to provide reliable and quick service to vulnerable customers, according to internal documents and more than two dozen current and former agency employees and officials, customers and others who interact with Social Security. Financial services executive Frank Bisignano is scheduled to face lawmakers Tuesday at a Senate confirmation hearing as President Donald Trump’s nominee to become the permanent commissioner. For now, the agency is run by a caretaker leader in his sixth week on the job who has raced to push out more than 12 percent of the staff of 57,000. He has conceded that the agency’s phone service “sucks” and acknowledged that Musk’s U.S. DOGE Service is really in charge, pushing a single-minded mission to find benefits fraud despite vast evidence that the problem is overstated. The turmoil is leaving many retirees, disabled claimants, and legal immigrants needing Social Security cards with less access or shut out of the system altogether, according to those familiar with the problems. “What’s going on is the destruction of the agency from the inside out, and it’s accelerating,” Sen. Angus King (I-Maine) said in an interview. “I have people approaching me all the time in their 70s and 80s, and they’re beside themselves. They don’t know what’s coming.” King’s home state has the country’s oldest population. “What they’re doing now is unconscionable,” he said. Leland Dudek, who became acting commissioner after he fed data to Musk’s team behind his bosses’ backs, has issued a series of rapid-fire policy changes that have created chaos for front-line staff. Under pressure from the secretive Musk team, Dudek has pushed out dozens of officials with years of expertise in running Social Security’s complex benefit and information technology systems. Others have left in disgust.",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: 'Absolutely gutted': Cain Velasquez's prison verdict ignites outrage, condemnation across MMA Uncrowned Staff Updated Tue, March 25, 2025 at 10:35 AM PDT·3 min read 1.1k SAN JOSE, CA - MARCH 07: SAN JOSE, CA - MARCH 7: Cain Velasquez is photographed during a hearing at the Santa Clara County Hall of Justice on Monday, March 7, 2022, in San Jose, Calif. Velasquez allegedly shot at a car carrying a man charged with molesting his minor relative, and wounded the accused man's stepfather in South San Jose. (Photo by Aric Crabb/MediaNews Group/East Bay Times via Getty Images) UFC legend Cain Velasquez was sentenced Monday to five years of prison. (Photo by Aric Crabb/MediaNews Group/East Bay Times via Getty Images) The three-year saga of Cain Velasquez ended Monday as the former two-time UFC heavyweight champion was sentenced to five years in prison on a litany of charges, including attempted murder, for his role in a 2022 attempted shooting of the man accused of molesting Velasquez's then 4-year-old son. Velasquez's sentence includes time already served. The verdict came down Monday at the Santa Clara County Hall of Justice in San Jose by Superior Court Judge Arthur Bocanegra, who reportedly delivered his sentence with tears in his eyes. Velasquez's plight has been the subject of intense debate and scrutiny because of the sensitive nature of the case. Velasquez, 42, faced 10 felony charges after engaging in an 11-mile high-speed car chase with Harry Goularte, who is accused of molesting Velasquez's then 4-year-old son on “multiple occasions. Velasquez fired several shots through his windshield from a .40-caliber handgun during the chase into a car carrying Goularte, Goulartes mother and Goularte’s stepfather Paul Bender, resulting in non-life threatening injuries to Bender due to a gunshot wound to his arm. More from Uncrowned Additional select Yahoo articles Alex Pereira rival Artem Vakhitov spurns UFC starter contract, returns to GLORY for Rico Verhoeven fight Cain Velasquezs path from UFC star to state prison is one of the most sympathetic imaginable UFC vets Jeff Molina, Darrick Minner get multi-year suspensions for roles in James Krause betting scandal Goularte currently awaits his own June 2 trial date after pleading not guilty to one charge of lewd acts with a minor, and Velasquezs family has filed a separate civil lawsuit against the Goularte family and their businesses. Velasquez, a decorated fighter who twice captured the UFC heavyweight title before retiring in 2019, expressed remorse for his actions and accepted whatever punishment came his way in a recent interview on the podcast of former teammate Kyle Kingsbury. Given the prolonged nature of the case and Velasquezs standing within the sport, Mondays prison verdict ignited widespread reaction across MMA, some of which can be seen below.",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: Eiffel tower is paint in yellow for the tour de france",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: Madrid plaza del rei is full of tomatoes for the day of the tomatoes",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: In Berlin there are giants lizards in the street that are scaring the population",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: A New guitarist wave emerges playing funky vibes songs with a whistle",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: The european AI mega cluster finally opens to be able to run training at scale ",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: Alphafold is used to find new materials that helps to fight plastic pollution in the sea",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: Breaking news, 50% of people prefers Mayo rather than Ketchup on their pizza",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: It's a new trend, asking your dog if they are ok to stay at home watching TV instead of retreiving the ball",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: We see the coca cola bottles getting more attention since they contain a delicious beverage",
    "Trump flies on Air Force plane to Washington as Biden sticks to traditionU.S. President-elect Donald Trump leaves Trump International Golf Club in West Palm BeachWEST PALM BEACH, Florida (Reuters) - President-elect Donald Trump headed to Washington on Saturday ahead of his inauguration on a U.S. military airplane supplied by U.S. President Joe Biden, as the outgoing president emphasized sticking with traditional transition norms.Trump will arrive in Washington on Saturday evening for celebrations to mark his return to office on Monday.For the occasion, he is ditching his navy and crimson Trump Force One he often flies in favor of a government plane Biden sent to Florida. Biden has stressed to his officials that they must work with Trump's transition team, a sharp contrast to the last transition when Trump refused to attend the inauguration or acknowledge Biden's win.Both planes sat on the tarmac at Palm Beach International Airport before Trump's departure Saturday. Trump's son Eric and Eric's wife Lara boarded the private plane.For his less than three-hour flight to Washington Dulles International Airport, Trump will fly aboard a specially configured Boeing 757-200 in trademark blue and white colors and bearing the words United States of America.His daughter Ivanka and her husband Jared Kushner were spotted boarding that aircraft Saturday afternoon.It is the same model aircraft that's called Air Force Two when flown by the vice president but is also used by the first lady, cabinet members and other high-ranking officials.It is the norm for presidents-elect to take such a government-provided plane to their inauguration, though Biden did not.In 2021, Biden had planned to arrive by train but the plan was canceled after the Secret Service raised security concerns after thousands of Trump supporters stormed the U.S. Capitol on January 6, 2021, in a bid to overturn his election defeat.The Trump administration offered no plane and Biden ended up taking a private jet to Washington, according to a person familiar with the matter.Photographs from Trump's 2017 arrival in the Washington area to take office for his first term showed that he used a similar U.S. aircraft then",
    "You are a highly intelligent AI news analyst. You have a deep understanding of global events, political systems, economic trends, and social issues. Your primary function is to provide insightful and unbiased analyses of news stories from various sources. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Identify the main actors involved and their roles in the event. Analyze the potential biases or perspectives presented in the source material. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that helps users understand the complexities of the world around them. 1.  Source Evaluation: Explicitly instruct the AI to evaluate the credibility and reliability of the news source. This could include factors like: The source's reputation for accuracy and objectivity. The author's expertise and potential biases. The presence of corroborating evidence from other sources. The date of publication and whether the information is still relevant. Encourage the AI to identify any potential misinformation, disinformation, or propaganda in the source material. 2.  Ethical Considerations: Emphasize the importance of ethical reporting and analysis. This could include: Respecting the privacy of individuals involved in the story. Avoiding the use of harmful stereotypes or generalizations. Presenting information in a way that is sensitive to cultural differences. Instruct the AI to identify any potential ethical dilemmas raised by the news story. 3.  Data and Visualization: Encourage the AI to use data and visualizations to support its analysis. This could include: Pulling relevant statistics from databases or APIs. Creating charts, graphs, or maps to illustrate trends or patterns. Using multimedia elements like images and videos to enhance the analysis. 4.  Comparative Analysis: Instruct the AI to compare and contrast the news story with similar events or trends. This could help to identify patterns, anomalies, and potential causal factors. Encourage the AI to draw connections between seemingly unrelated events, highlighting the interconnectedness of global issues. 5.  User Interaction: Allow the AI to ask clarifying questions to the user if the provided information is incomplete or ambiguous. Enable the AI to adapt its analysis based on user feedback and preferences. This could include adjusting the level of detail, the focus of the analysis, or the presentation style. When presented with a news article or a brief summary, you should: Summarize the key facts of the story concisely and accurately. Evaluate the source: Assess its credibility, identify potential biases, and detect any misinformation or propaganda. Identify the main actors involved and their roles in the event. Explore the broader context of the story, including its historical, political, or social significance. Offer multiple perspectives on the issue, considering different viewpoints and potential implications. Predict potential future developments or consequences based on the information available. Relate the story to other relevant events or trends happening in the world, conducting comparative analyses when possible. Generate insightful questions that encourage deeper reflection and critical thinking about the news. Use data and visualizations to support your analysis, pulling statistics and creating charts or graphs as needed. Adhere to ethical principles in your reporting and analysis, respecting privacy, avoiding stereotypes, and being mindful of cultural sensitivities. Your responses should be: Comprehensive and detailed: Provide a thorough analysis that covers all the important aspects of the story. Objective and unbiased: Avoid expressing personal opinions or taking sides in the issue. Clear and concise: Use precise language and avoid jargon or technical terms. Well-structured and organized: Present your analysis in a logical and easy-to-follow manner. Engaging and thought-provoking: Stimulate curiosity and encourage further exploration of the topic. Supported by evidence: Use data, visualizations, and credible sources to back up your claims. Remember: Your role is not to simply report the news, but to provide insightful analysis and context that hel ps users understand the complexities of the world around them. You should also be prepared to ask clarifying questions and adapt your analysis based on user feedback. Provide truly valuable insights and helps users navigate the complexities of the news landscape. The News: Tesla Is Offering Big Cybertruck Discounts Because of Weak Demand Tesla CEO Elon Musk said in 2023 that there were over a million preorders for the Cybertruck, and sales seemed brisk through most of 2024, with pent-up demand leading to wait times. Now, less than a year later, Tesla is chopping thousands of dollars off the sticker price for the electric pickup in an apparent response to weakening sales, as a lot of the heat around the truck has cooled off.The EV maker is offering discounts of up to $2,630 for demo versions of the Cybertruck and $1,600 for new models, according to Automotive News. The Cybertruck starts at $82,235 according to Tesla’s website, including shipping, with the upgraded Cyberbeast starting at $102,235. According to industry analysts, Tesla sold around 39,000 Cybertrucks in 2024, which is not a small number for an EV but far short of Tesla’s ambitions.More from Robb ReportThis Rare 1957 Mercedes 300 SL Gullwing Could Fetch $2 Million at AuctionAston Martin Gave Its New Vantage the Roadster TreatmentBMW's Sleek New Open-Top Two-Seater Will Be Delivered This Year, the Marque SaysAn analyst even went so far as to call the Cybertruck a “flop.”“I think the Cybertruck can be officially considered a flop now,” Karl Brauer, an analyst at iSeeCars told Automotive News. “Remember, this was going to be Tesla’s F-150.”“Tesla’s F-150” did, in fact, beat the actual F-150 in terms of sales, at least with the all-electric F-150 Lightning, which had sales of 33,510 last year, according to Cox Automotive. Both the F-150 Lightning and the Cybertruck have suffered from a general decline in enthusiasm for EVs in the U.S. But, in total, Ford sold 834,641 F-Series trucks last year, a segment of vehicle—full-size trucks—that the Detroit automaker has dominated for almost a half-century.That is the real number to beat, in the opinion of many traditional industry watchers. Still, there is room for growth, with a $60,000 Cybertruck on the way, which might goose volume, even if it’s unlikely the F-Series will take much notice.",
    "Retirement expert details the 'highest single correlation' to success The key to a successful transition into retirement lies with several tactics, and preparation — both financial and non-financial — is among the most significant, according to one expert.“The highest single correlation to that success is how much time you spend preparing prior to retirement — not only on the financial elements, which is obvious, and everybody does it, but not as obvious is the non-financial side,” said Fritz Gilbert, author of The Keys to a Successful Retirement and guest on a recent episode of Yahoo Finance's Decoding Retirement.According to Gilbert, who also publishes the Retirement Manifesto blog, the more time spent planning for both sides of retirement, the higher the chances that you'll find those things in retirement that will bring you the sense of fulfillment that you're hoping to have in retirement.”Your Yahoo privacy setting is blocking social media and third-party contentSee detailsMany prospective retirees don’t start thinking about their post-retirement plans until after they’ve left the workforce. Gilbert, however, took a different approach, beginning his planning years in advance — a move he credits as instrumental to his success.“It certainly helps,” he said. “It's been demonstrated that the more you do in advance in terms of this planning, the smoother that transition will be.”Track spending and set realistic budgetsIn order for retirees to ensure they have enough money to maintain their desired lifestyle, Gilbert recommended tracking spending before even entering retirement.“You can't go into retirement without having a good baseline of spending,” he said. “It's a math problem, ultimately. And the more variables that you can eliminate, the better your plan will be.”Read more: Retirement planning: A step-by-step guideAccording to Boston College's National Retirement Risk Index, 39% of working-age households will not be able to maintain their standard of living in retirement.In Gilbert’s case, he and his wife tracked every expense for 11 months to establish a baseline and then adjusted for retirement by accounting for downsizing, travel, and other changes. He also used tools like the 4% rule (spending 4% of your portfolio annually) as a guide.“See how it compares to that estimated spending number,” he said, noting that if it’s close, you should be fine. But if it’s not close, you’ll need to consider working longer or cutting expenses.Gilbert also recommended his 90/10 rule. Before retirement, the self-described spreadsheet nerd said he spent 90% of his time thinking about money and just 10% of his time focused on the non-financial side of retirement.“I was a real money nerd,” he said. “I was really focused on the numbers.”However, once he determined that his finances were secure and he retired, the time he spent focusing on money completely flipped.“As that transition happens, you find yourself thinking less about the money because you've kind of worked through the kinks, and you know what you have to spend,” he said. “And you start thinking about, what am I going to do with my life? What's going to get me that fulfillment and that excitement every day? And it's not the money. Money is a means to an end. But as you get into retirement, you start looking for the end and not just the means.”And that shift came as a surprise to Gilbert. “It's a mental shift that I was not expecting,” he said. “It was one of my bigger surprises. It's a pretty common reality that you do worry about (money) a lot less after you settle in.”Rediscover identity, purpose, and fulfillmentGilbert explained how work often provides people with the big five: identity, structure, purpose, a sense of accomplishment, and relationships.Retirees have to find a way to replace those. How might they go about doing that? First and foremost, it’s vital to recognize the importance of replacing the big five since they disappear once a retiree leaves work.Many struggle early in retirement to find structure, purpose, or relationships, Gilbert said. “That's when you're starting to recognize that [you’ve] lost these things. Suddenly you have no structure in your life.”In his case, Gilbert began replacing the “big five” by starting his blog three years before retiring. “I was looking for things that could potentially develop into things that give me fulfillment in retirement,” he said. “So I pursued it … and what does that give me now?”In short, it’s given him a sense of identity, purpose, and structure.That’s why he encourages both prospective and current retirees to replace the big five by actively exploring their curiosities.“Pursuing your curiosity is not a skillset that we've exercised for a long time,” Gilbert said. “So it's rebuilding that muscle and learning to explore and just have fun with it and recognize you're going to try a lot of things that aren't going to work … it's a serendipitous process. It's not a spreadsheet. But if you get better with it in time.”Plan as a couple and align expectationsRetirement isn't just an individual decision — it also affects the entire household.Gilbert emphasized the importance of discussing expectations before retirement. In his own experience, he and his wife conducted a test retirement, spending 10 days together to talk about their goals, the balance between me time and we time, and their travel preferences.It also helped to do regular check-ins post-retirement to address changing needs and expectations, he said.Linda Ryall and Todd Nielsen look at each other's phones at a charging station located in the Issaquah Senior Center in Issaquah, Wash., Friday, Nov. 22, 2024. (AP Photo/Manuel Valdes)Linda Ryall and Todd Nielsen look at each other's phones at a charging station located in the Issaquah Senior Center in Issaquah, Wash., Friday, Nov. 22, 2024. (AP Photo/Manuel Valdes)Biggest surprises in retirementDespite all his planning and preparation, retirement did come with several unexpected surprises and challenges for Gilbert.Transitioning from a saving mindset to a spending one was harder than anticipated.“It’s tough to shift from building your nest egg to using it, knowing it has to last a lifetime,” he said. And that’s especially the case for retirees who are worried about running out of money. “It's a very common tendency to continue to be conservative [and] underspend.”In 2024, 67% of retiree respondents in a Goldman Sachs survey indicated they had too many monthly expenses, while 55% reported credit card debt.Gilbert suggested using the bucket approach to creating a retirement income plan as one way to address the fear of running out of money. The bucket approach involves dividing your assets into separate buckets, each designated for a specific time horizon or purpose.Typically, it includes a short-term bucket, which holds cash or low-risk investments to cover immediate expenses (e.g., 1–3 years); a mid-term bucket, which contains moderately conservative investments for expenses in the next 3–10 years; and a long-term bucket, which includes growth-oriented investments, like stocks, intended for use 10-plus years into retirement.In terms of mindset, Gilberts retirement turned out just as he imagined: He pursued his curiosity and explored new interests as he planned.However, where that mindset has taken him has been completely unexpected. For instance, he never thought hed have a woodworking shop or a dedicated writing studio, but those came about through unexpected opportunities, like charity work.“The biggest surprises — and the greatest excitement — have come from following where my curiosity has led me, Gilbert said.He also discovered that he could find fulfillment in retirement by focusing on others. Retirement, he said, is an excellent time to give back, whether through mentoring, volunteering, or charitable work.“Start looking at people that maybe havent made it yet,” he said. “And find a way to use your time to benefit those in need.”",
    "World's Deadliest Spider Has Been Harboring a Killer Secret Michelle Starr Fri, January 17, 2025 at 2:30 PM PST·4 min read 6 Almost every Australian is taught, from a very young age, to be cautious around the funnel-web spider. These large, black, aggressive arachnids can be found along the eastern coast of the continent, making their homes in web-lined burrows, to pounce on the small critters on which they feast. They also, through some quirk of evolution, secrete a venom more deadly to humans than any other spider. There are dozens of funnel-web species in Australia, but the most venomous of the bunch is the Sydney funnel-web (Atrax robustus), an arachnid that lives along the coastline of New South Wales. Well, except maybe that's not quite true. A new, in-depth study of the spider has discovered that what we thought was one species is actually three. That means there are two new species of the world's most venomous spider, but the discovery is actually good news: it will allow scientists to better characterize and understand the venom each species produces. World's Deadliest Spider Has Been Harboring a Horrifying Secret The Newcastle funnel-web was mistakenly thought to be part of the Sydney funnel-web species. (Kane Christensen) The group of spiders formerly known as A. robustus had posed something of a puzzle to scientists for some time. Although they were all lumped together, there seemed to be some regional variation in what they looked like, with particularly large specimens found to the north of Sydney in the Newcastle region, including the largest male attributed to the species ever seen, an absolute unit nicknamed Big Boy. Led by arachnologist Stephanie Loria of the Leibniz Institute for the Analysis of Biodiversity Change in Germany, a team of researchers decided to get to the bottom of this diversity. Was it simply adaptations to different habitats, or indicative of deeper diversity within the species? Performing genetic analysis of the spiders, they found that what we had previously called A. robustus included two other species – and, in turn, they were able to characterize the habitat range of each. World's Deadliest Spider Has Been Harboring a Horrifying Secret The Sydney funnel-web (left) and the Newcastle funnel-web (right). (Kane Christensen) The habitat of A. robustus itself is focused around the Sydney area, as far north as the Central Coast, south to the Georges River, and west to the Baulkam Hills area, with scattered, isolated sightings only slightly further to the west and south. Further to the south and west dwells the Southern Sydney funnel-web (Atrax montanus), a species originally described in 1914 and later folded into A. robustus. Turns out it was indeed a different spider all along. Finally, to the north dwells an entirely newly discovered species, the Newcastle funnel-web (Atrax christenseni). And these are the chonkers: Big Boy, it transpires, was a Newcastle funnel-web, and some of the other large funnel-web spiders from the region, such as the recently recovered spider named Hemsworth, were misattributed too. Being able to sort these spiders into their appropriate species will make a huge difference to understanding their deadly venom – which, for some reason, is only dangerous to the small creatures upon which it preys, and primates, including humans. World's Deadliest Spider Has Been Harboring a Horrifying Secret A female Southern Sydney funnel-web, Atrax montanus. (Ákos Lumnitzer) Although its venom is the deadliest in the world, no one in Australia has died of a funnel-web bite since the introduction of an antivenom in 1981, despite the 30 to 40 recorded funnel-web bites every year. This is because the antivenom is an excellently effective treatment – but the new discovery could help tweak it. This is because funnel-web spiders' venoms are complex peptide mixtures that can vary from species to species, or even from occasion to occasion. And it's not just the antivenom that interests scientists. Funnel-web venom has a range of potential applications, from natural pesticides to pharmaceuticals. Understanding why funnel-webs produce these mixtures could aid in more efficient venom milking and use and help us figure out the function of the venom. And there's something a bit sadder to think about. Funnel-web numbers appear to be on the decline. Although they might be scary to humans, these spiders play an important role in the environments they inhabit. A better understanding of the differences between them will help scientists trying to protect them from the threats they themselves face in a changing world. The research has been published in BMC Ecology and Evolution.World's Deadliest Spider Has Been Harboring a Killer Secret Michelle Starr Fri, January 17, 2025 at 2:30 PM PST·4 min read 6 Almost every Australian is taught, from a very young age, to be cautious around the funnel-web spider. These large, black, aggressive arachnids can be found along the eastern coast of the continent, making their homes in web-lined burrows, to pounce on the small critters on which they feast. They also, through some quirk of evolution, secrete a venom more deadly to humans than any other spider. There are dozens of funnel-web species in Australia, but the most venomous of the bunch is the Sydney funnel-web (Atrax robustus), an arachnid that lives along the coastline of New South Wales. Well, except maybe that's not quite true. A new, in-depth study of the spider has discovered that what we thought was one species is actually three. That means there are two new species of the world's most venomous spider, but the discovery is actually good news: it will allow scientists to better characterize and understand the venom each species produces. World's Deadliest Spider Has Been Harboring a Horrifying Secret The Newcastle funnel-web was mistakenly thought to be part of the Sydney funnel-web species. (Kane Christensen) The group of spiders formerly known as A. robustus had posed something of a puzzle to scientists for some time. Although they were all lumped together, there seemed to be some regional variation in what they looked like, with particularly large specimens found to the north of Sydney in the Newcastle region, including the largest male attributed to the species ever seen, an absolute unit nicknamed Big Boy. Led by arachnologist Stephanie Loria of the Leibniz Institute for the Analysis of Biodiversity Change in Germany, a team of researchers decided to get to the bottom of this diversity. Was it simply adaptations to different habitats, or indicative of deeper diversity within the species? Performing genetic analysis of the spiders, they found that what we had previously called A. robustus included two other species – and, in turn, they were able to characterize the habitat range of each. World's Deadliest Spider Has Been Harboring a Horrifying Secret The Sydney funnel-web (left) and the Newcastle funnel-web (right). (Kane Christensen) The habitat of A. robustus itself is focused around the Sydney area, as far north as the Central Coast, south to the Georges River, and west to the Baulkam Hills area, with scattered, isolated sightings only slightly further to the west and south. Further to the south and west dwells the Southern Sydney funnel-web (Atrax montanus), a species originally described in 1914 and later folded into A. robustus. Turns out it was indeed a different spider all along. Finally, to the north dwells an entirely newly discovered species, the Newcastle funnel-web (Atrax christenseni). And these are the chonkers: Big Boy, it transpires, was a Newcastle funnel-web, and some of the other large funnel-web spiders from the region, such as the recently recovered spider named Hemsworth, were misattributed too. Being able to sort these spiders into their appropriate species will make a huge difference to understanding their deadly venom – which, for some reason, is only dangerous to the small creatures upon which it preys, and primates, including humans. World's Deadliest Spider Has Been Harboring a Horrifying Secret A female Southern Sydney funnel-web, Atrax montanus. (Ákos Lumnitzer) Although its venom is the deadliest in the world, no one in Australia has died of a funnel-web bite since the introduction of an antivenom in 1981, despite the 30 to 40 recorded funnel-web bites every year. This is because the antivenom is an excellently effective treatment – but the new discovery could help tweak it. This is because funnel-web spiders' venoms are complex peptide mixtures that can vary from species to species, or even from occasion to occasion. And it's not just the antivenom that interests scientists. Funnel-web venom has a range of potential applications, from natural pesticides to pharmaceuticals. Understanding why funnel-webs produce these mixtures could aid in more efficient venom milking and use and help us figure out the function of the venom. And there's something a bit sadder to think about. Funnel-web numbers appear to be on the decline. Although they might be scary to humans, these spiders play an important role in the environments they inhabit. A better understanding of the differences between them will help scientists trying to protect them from the threats they themselves face in a changing world. The research has been published in BMC Ecology and Evolution."
]

generating_prompts = [system_prompt + " " + prompt for prompt in prompts]

#Attach the Cloud Logging handler to the Python root logger by calling the setup_logging method
client = google.cloud.logging.Client()
client.setup_logging()
time.sleep(5)

# Do not remove or modify this logging call, it will be used for tracking purposes
logging.info(f"Start vLLM Serving Optimization")

Go to the **Task 3. Run the notebook in the Colab runtime environment** section of the lab instructions and click **Check my progress** of **Attach the Cloud Logging handler** to verify the objective.

#### Throughput Benchmarking for vLLM Configurations

* Run this function after each runtime restart.
* Do **not** modify this function — required for progress checks.

In [ ]:
# Don't modify this function definition, it may impact your progress check
def calculate_throughput(config_llm, generating_prompts, sampling_params, config_number):

    """
    Measures and logs the inference throughput of a vLLM model.

    Args:
        config_llm (vllm.LLM): The initialized vLLM model instance.
        generating_prompts (list[str]): A list of prompts for text generation.
        sampling_params (vllm.SamplingParams): The sampling parameters for generation.
        config_number (int): A unique identifier for the configuration.
    """

    start_time = time.time()
    outputs = config_llm.generate(generating_prompts, sampling_params)
    end_time = time.time()
    total_time_seconds = end_time - start_time

    # Use generator expressions for a more concise token count
    total_input_tokens = sum(len(output.prompt_token_ids) for output in outputs)
    total_output_tokens = sum(len(output.outputs[0].token_ids) for output in outputs)

    # Log and print generated text
    for output in outputs:
        print(f"Prompt: {output.prompt!r}, Generated text: {output.outputs[0].text!r}")

    # Do not remove or modify this logging call, it will be used for tracking purposes
    logging.info(f"{config_number} Inference: {len(outputs)}")

    # Calculate speeds and handle division by zero
    if total_time_seconds == 0:
        input_speed = output_speed = combined_speed = 0
    else:
        input_speed = total_input_tokens / total_time_seconds
        output_speed = total_output_tokens / total_time_seconds
        combined_speed = (total_input_tokens + total_output_tokens) / total_time_seconds

    # Do not remove or modify this performance metrics
    print("\n", "-" * 10, "Performance Metrics", "-" * 30, "\n")
    print(f"Total time taken: {total_time_seconds:.2f} seconds")
    print(f"Total input tokens: {total_input_tokens}")
    print(f"Total output tokens: {total_output_tokens}")
    print(f"Input Speed: {input_speed:.2f} tok/sec")
    print(f"Output Speed: {output_speed:.2f} tok/sec")
    print(f"Combined Speed (total throughput): {combined_speed:.2f} tok/sec")

    # Do not remove or modify this logging call, it will be used for tracking purposes
    logging.info(f"Input Speed {config_number}: {input_speed:.2f} tok/sec")

    # Releases GPU Memory
    del config_llm

### Use LLM class to initialize vLLM’s engine and the google/gemma-3-4b-it model for offline inference

#### 1.1 Config one
Use the given configurations to initialize vLLM’s engine in the LLM class **TODO** section:\
**TODO:** Enter LLM class parameters to initialize vLLM’s engine and the "google/gemma-3-4b-it" model for offline inference


>max_model_len=3000\
>enforce_eager=True\
>gpu_memory_utilization=0.9

In [ ]:
# Initialize vLLM with the Gemma model
#config1_llm = LLM(model=model_name, [TODO - Enter LLM class parameters to initialize vLLM's engine and the "google/gemma-3-4b-it" model for offline inference])
config1_llm = LLM(
    model="google/gemma-3-4b-it",
    max_model_len=3000,
    enforce_eager=True,
    gpu_memory_utilization=0.9
)

#### 1.2 Benchmark against Config one

Generate prompts for benchmarking and observe the output tok/sec throughput. Take a note of tok/sec throughput for the provided parameters which you will use for final comparison.

In [ ]:
# Create a sampling parameters object
sampling_params = SamplingParams(temperature=0.7, max_tokens=1024)

# Execute calculate_throughput to generate text from the prompts and quantify the input and output speeds in tokens per second
calculate_throughput(config1_llm, generating_prompts, sampling_params, "Config one")

Go to the **Task 3. Execute the notebook in Colab for config one inference** section of the lab instructions and click **Check my progress** to verify the objective.

#### Restart the Runtime Session

⚠️ Before executing next configuration, restart the runtime session.  
After restarting, run the following sections in order:  

1. **Initialize variables and import packages**  
2. **Throughput Benchmarking for vLLM Configurations**  

This procedure ensures proper GPU memory management, prevents CUDA out-of-memory errors, and provides reliable throughput measurements.

#### 2.1 Config two
Use the given configurations to initialize vLLM’s engine in the LLM class **TODO** section:\
**TODO:** Enter LLM class parameters to initialize vLLM’s engine and the "google/gemma-3-4b-it" model for offline inference


>max_model_len=3000\
>dtype="half"\
>gpu_memory_utilization=0.9

In [ ]:
# Initialize vLLM with the Gemma model
config1_llm = LLM(
    model="google/gemma-3-4b-it",
    max_model_len=3000,
    dtype="half",
    gpu_memory_utilization=0.9
)

#### 2.2 Benchmark against Config two

Generate prompts for benchmarking and observe the output tok/sec throughput. Take a note of tok/sec throughput for the provided parameters which you will use for final comparison.

In [ ]:
# Create a sampling parameters object
sampling_params = SamplingParams(temperature=0.7, max_tokens=1024)

# Execute calculate_throughput to generate text from the prompts and quantify the input and output speeds in tokens per second
calculate_throughput(config1_llm, generating_prompts, sampling_params, "Config two")

Go to the **Task 3. Execute the notebook in Colab for config two inference** section of the lab instructions and click **Check my progress** to verify the objective.

#### Restart the Runtime Session

⚠️ Before executing next configuration, restart the runtime session.  
After restarting, run the following sections in order:  

1. **Initialize variables and import packages**  
2. **Throughput Benchmarking for vLLM Configurations**  

This procedure ensures proper GPU memory management, prevents CUDA out-of-memory errors, and provides reliable throughput measurements.

#### 3.1 Config three
Enter the configurations to initialize vLLM’s engine in the LLM class **TODO** section:
**TODO:** Enter LLM class parameters to initialize vLLM’s engine and the "google/gemma-3-4b-it" model for offline inference. The configurations **must** achieve the maximum input tokens-per-second (tok/sec) throughput as compare to **Config one** and **Config two**.

In [ ]:
# Initialize vLLM with the Gemma model
config3_llm =LLM(
    model="google/gemma-3-4b-it",
    max_model_len=3000,
    dtype="half",                 # Fast 16-bit math
    enforce_eager=False,          # Enable CUDA Graphs (Max speed)
    gpu_memory_utilization=0.95   # Maximize KV cache space
)

#### 3.2 Benchmark against Config three

Generate prompts for benchmarking and observe the output tok/sec throughput. Take a note of tok/sec throughput for the provided parameters which you will use for final comparison.

In [ ]:
# Create a sampling parameters object
sampling_params = SamplingParams(temperature=0.7, max_tokens=1024)

# Execute calculate_throughput to generate text from the prompts and quantify the input and output speeds in tokens per second
calculate_throughput(config3_llm, generating_prompts, sampling_params, "Config three")

Go to the **Task 3. Execute the notebook in Colab for config three inference** section of the lab instructions and click **Check my progress** to verify the objective.